# Roteiro de Aula: Modelos Ensemble com Qualidade de Vinhos

Este laboratorio aplica Bagging, Random Forest, Boosting, Stacking e Voting para classificar a qualidade de vinhos a partir de atributos fisico-quimicos. A pratica relaciona os resultados aos conceitos de vies, variancia, diversidade e custo computacional.

**Base:** `vinhos.csv`  
**Tarefa:** classificar a qualidade em `ruim` (nota <= 5), `medio` (nota = 6) e `bom` (nota >= 7).

## Objetivos
- Preparar dados sem vazamento entre treino e teste.
- Estabelecer um modelo baseline.
- Comparar Bagging, Random Forest e Extra Trees como metodos de reducao de variancia.
- Comparar AdaBoost e Gradient Boosting como metodos sequenciais de reducao de vies.
- Construir ensembles por Stacking e Voting.
- Usar um placar simples para observar comportamento, sem buscar uma otimizacao de metricas nesta aula.

## 1. Carregamento e analise inicial
1. Importe as bibliotecas necessarias.
2. Carregue `vinhos.csv` em `df`.
3. Inspecione dimensoes, tipos, valores ausentes, duplicatas e a distribuicao de `qualidade`.

### Perguntas
- Quais variaveis possuem valores ausentes?
- A distribuicao da qualidade sugere classes equilibradas?
- Por que a propria coluna `qualidade` nao pode ser usada como atributo?

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, RandomForestClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
sns.set_theme(style='whitegrid')

df = pd.read_csv('vinhos.csv')
print('Dimensoes:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False))
print('Duplicatas:', df.duplicated().sum())
display(df['qualidade'].value_counts().sort_index())

## 2. Preparacao da tarefa
Remova duplicatas, crie a variavel alvo `categoria` e divida os dados em treino e teste. A divisao deve ser estratificada para preservar a proporcao das categorias.

O pre-processamento sera ajustado depois, dentro do pipeline de cada modelo. Assim, as medianas, escalas e categorias aprendidas dependem somente do conjunto de treino.

In [ ]:
df_model = df.drop_duplicates().copy()

def categorizar_qualidade(nota):
    if nota <= 5:
        return 'ruim'
    if nota == 6:
        return 'medio'
    return 'bom'

df_model['categoria'] = df_model['qualidade'].apply(categorizar_qualidade)

# TODO: crie feature_cols excluindo qualidade e categoria.
# TODO: defina X e y e realize train_test_split com test_size=0.25,
#       random_state=SEED e stratify=y.

# feature_cols = ...
# X = ...
# y = ...
# X_train, X_test, y_train, y_test = ...

display(df_model['categoria'].value_counts(normalize=True).sort_index())

## 3. Pre-processamento e baseline
1. Identifique as colunas numericas e a coluna categorica `lote`.
2. Crie um `ColumnTransformer` que impute a mediana e escale as variaveis numericas, e que impute a moda e aplique One-Hot Encoding em `lote`.
3. Crie uma funcao `avaliar_modelo` que registre um placar simples e o tempo de treino.
4. Use uma arvore de decisao como baseline.

### Placar ludico
Pense na acuracia como a proporcao de palpites certos de um modelo em um jogo. O F1 macro apenas garante que cada uma das tres categorias tenha o mesmo peso nesse placar, mesmo que uma apareca menos vezes. Nesta pratica, os numeros servem para acompanhar o experimento, nao para eleger um campeao definitivo.

### Perguntas
- O que esse placar revela sobre os palpites do modelo?
- O que o pipeline impede que ocorra com os dados de teste?

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = ['lote']

def criar_preprocessador():
    # TODO: retorne um ColumnTransformer com os dois fluxos descritos acima.
    pass

resultados = []

def avaliar_modelo(nome, modelo, categoria):
    # TODO: meca o tempo, ajuste o modelo, gere predicoes no teste
    #       e inclua nome, categoria, acuracia, f1_macro e tempo em resultados.
    pass

# TODO: crie um Pipeline com criar_preprocessador() e DecisionTreeClassifier.
# baseline = ...
# avaliar_modelo('Arvore de Decisao', baseline, 'Baseline')

## 4. Bagging: reducao de variancia
Treine tres ensembles paralelos baseados em arvores: `BaggingClassifier`, `RandomForestClassifier` e `ExtraTreesClassifier`. Use 150 estimadores, `random_state=SEED` e `n_jobs=-1` quando disponivel.

### Perguntas
- Qual diferenca existe entre Bagging e Random Forest quanto a selecao de atributos?
- Por que combinar arvores pouco correlacionadas tende a reduzir a variancia?
- Qual modelo teve o melhor equilibrio entre F1 macro e tempo?

In [ ]:
# TODO: crie e avalie os tres modelos em Pipelines independentes.
# bagging = Pipeline([...])
# random_forest = Pipeline([...])
# extra_trees = Pipeline([...])
# avaliar_modelo('Bagging', bagging, 'Bagging')
# avaliar_modelo('Random Forest', random_forest, 'Bagging')
# avaliar_modelo('Extra Trees', extra_trees, 'Bagging')

## 5. Boosting: reducao de vies
Treine `AdaBoostClassifier` com arvores rasas (`max_depth=2`) e `GradientBoostingClassifier`. Use 150 estimadores e `learning_rate=0.05`.

### Perguntas
- Por que o boosting treina modelos sequencialmente?
- Qual o efeito esperado de diminuir `learning_rate` e aumentar `n_estimators`?
- Por que boosting pode ser mais sensivel a ruido e outliers?

In [ ]:
# TODO: crie pipelines para AdaBoost e Gradient Boosting e avalie-os.
# Dica: no AdaBoost, use estimator=DecisionTreeClassifier(max_depth=2, random_state=SEED).
# adaboost = ...
# gradient_boosting = ...
# avaliar_modelo('AdaBoost', adaboost, 'Boosting')
# avaliar_modelo('Gradient Boosting', gradient_boosting, 'Boosting')

## 6. Stacking e Voting
Monte modelos diversos para combinar suas previsoes. No Stacking, use Random Forest, KNN e Regressao Logistica como estimadores base e Regressao Logistica como meta-modelo. No Voting, compare as variantes `hard` e `soft` com Random Forest, Gradient Boosting e Regressao Logistica.

### Perguntas
- Qual informacao chega ao meta-modelo do Stacking?
- Por que os modelos base devem ser diversificados?
- Quando o soft voting pode ser preferivel ao hard voting?

In [ ]:
# TODO: crie StackingClassifier(..., cv=5, n_jobs=-1) dentro de um Pipeline.
# TODO: crie VotingClassifier para hard e soft voting dentro de Pipelines.
# avaliar_modelo('Stacking', stacking, 'Stacking')
# avaliar_modelo('Hard Voting', voting_hard, 'Voting')
# avaliar_modelo('Soft Voting', voting_soft, 'Voting')

## 7. Comparacao final e interpretacao
Organize o placar apenas para observar como cada familia de ensemble se comportou. Em seguida, gere a matriz de confusao do modelo com maior F1 macro como um mapa dos acertos e confusoes entre as categorias.

### Conclusao
Registre: como os modelos se comportaram no placar; a relacao entre tempo e resultado; evidencias de reducao de vies ou variancia; e por que uma escolha real de modelo exigiria mais analise do que uma unica rodada de metricas.

In [ ]:
# TODO: transforme resultados em DataFrame, ordene por F1 macro e visualize o ranking.
# TODO: recupere ou retreine o melhor modelo e exiba ConfusionMatrixDisplay.

# df_resultados = pd.DataFrame(resultados).sort_values('F1 macro', ascending=False)
# display(df_resultados)
# ...

## 8. Desafio extra: categorias nativas com CatBoost

Agora use `bank-vf.csv`, uma base de marketing bancario com varias variaveis categoricas, para prever `y` (`yes` ou `no`). Compare Random Forest, LightGBM e CatBoost.

- Random Forest e LightGBM receberao categorias codificadas por One-Hot Encoding.
- CatBoost recebera as categorias diretamente, sem One-Hot Encoding.
- Uma amostra de 12 mil registros deixa o experimento mais rapido e reproduzivel.

Instale os pacotes extras no kernel se necessario:

```python
%pip install lightgbm catboost
```

### Perguntas
- Quais colunas sao categoricas nessa base?
- Por que o CatBoost pode receber essas colunas sem codificacao One-Hot?
- O que muda quando o mesmo tipo de dado passa por preparacoes diferentes?

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# TODO: carregue bank-vf.csv e remova colunas de indice, se existirem.
# TODO: use uma amostra de ate 12000 registros com random_state=SEED.
# TODO: transforme y em 0 (no) e 1 (yes), crie X_bank e divida treino/teste com stratify.
# TODO: separe as colunas numericas e categoricas.

# df_bank = ...
# y_bank = ...
# X_bank_train, X_bank_test, y_bank_train, y_bank_test = ...
# numeric_bank = ...
# categorical_bank = ...

print('Prepare a base bancaria e identifique as colunas categoricas.')

### 8.1 Pipelines e relogio do experimento

Crie tres pipelines:

1. Random Forest com imputacao numerica e One-Hot Encoding das categorias.
2. LightGBM com o mesmo pre-processamento do Random Forest.
3. CatBoost com as categorias preenchidas por um rotulo de ausencia e passadas diretamente ao estimador.

Para cada modelo, meca o tempo de treino e o tempo de escoragem (o tempo para gerar palpites no teste). Use acuracia e F1 macro somente como o placar do experimento.

### Perguntas
- Qual modelo levou mais tempo para aprender?
- Qual foi mais rapido para dar palpites?
- O placar e os tempos contam a mesma historia?

In [ ]:
# TODO: crie o pre-processador One-Hot para Random Forest e LightGBM.
# TODO: crie uma funcao preparar_catboost que preencha categorias ausentes com '__ausente__'.
# TODO: monte os tres pipelines e uma funcao para medir treino, escoragem e placar.

# modelos_bank = {
#     'Random Forest': ...,
#     'LightGBM': ...,
#     'CatBoost': ...
# }
# resultados_bank = []
# for nome, modelo in modelos_bank.items():
#     ...

# TODO: monte um DataFrame com Modelo, Acuracia, F1 macro,
#       Tempo de treino (s) e Tempo de escoragem (s).
print('Compare o relogio e o placar dos tres modelos.')